HumanInTheLoopMiddleware（人在环中间件、人工审核中间件）在 `工具调用前` 中断Agent运行，等待用户对工具调用请求决策。可选的决策有： `approve（同意执行）` 、 `edit（编辑调用配置后执行）` 、` reject（拒绝执行）` 。

In [1]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

#从.env文件中加载环境变量
load_dotenv(override=True)

main_model = init_chat_model(
    model="qwen3.7-flash",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
)


举例1：调用前中断

In [11]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware,InterruptOnConfig
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig
from langchain.messages import HumanMessage

from langchain.tools import tool
from rich import print as rprint


@tool(name_or_callable="get_weather_tool")
def get_weather(city: str, is_forcast: bool = False) -> str:
    """
    查询指定城市天气

    Args:
        city: 城市名称
        is_forcast: 是否包含明天天气预报？

    Returns:
        指定城市天气
    """
    res = f"{city}今天天气不错"
    if is_forcast:
        res += "\n明天下雨"
    return res

@tool(name_or_callable="get_news_tool")
def get_news() -> str:
    """
    查询当日新闻

    Returns:
        中方三艘邮轮通过霍尔木兹海峡
    """
    return "中方三艘邮轮通过霍尔木兹海峡"

@tool(name_or_callable="read_email_tool")
def read_email(email_id: str) -> str:
    """
    通过邮件ID读取内容的伪函数

    Args:
        email_id: 邮件ID

    Returns:
        邮件ID:xxxx
    """
    return f"邮件ID:{email_id}\n是空的"

@tool
def send_email(recipient:str, subject:str, content:str) -> str:
    """
    发送邮件伪函数

    Args:
        recipient: 邮件接收人
        subject: 邮件主题
        content: 邮件内容

    Returns:
        发给邮件的内容信息
    """
    print(">>> 真的执行发送邮件工具了")
    return f"发送给{recipient}的邮件标题是：{subject}，内容是：{content}"

new_agent = create_agent(
    model=main_model,
    tools=[get_weather, get_news, read_email, send_email],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "get_weather_tool": True,
                "get_news_tool": True,
                "read_email_tool": False,
                "send_email": InterruptOnConfig(
                     allowed_decisions = ["approve", "reject"],
                     description = "发送邮件中断啦"
                ),
            },
            description_prefix="中断咯"
        ),
    ],
)

custom_config: RunnableConfig = {"configurable":{"thread_id": "1"}}

#第一次调用：会暂停在发送邮件前
response = new_agent.invoke(
    {
        "messages": [
            HumanMessage(content="请帮我查询今天深圳的天气"
                         "查询今天的新闻"
                         "查看ID为 'SK34132313210'的邮件内容"
                         "向boy@qq.com发送邮件，标题是'关于AI编程',内容是: '你好，人工智能。'"
                         "同时做这四件事"
                         )
        ]
    },
    config = custom_config,
)

print("==== 第一次 invoke 返回 ====")
print("========= 原始响应 =========")
rprint(response)
print("========= 美化输出 =========")
for msg in response["messages"]:
    msg.pretty_print()

#关键： 看中断信息
interrupts = response.get("__interrupt__", [])
print("============ interrupts ==============")
rprint(interrupts)

action_requests = interrupts[0].value["action_requests"]

for action_request in action_requests:
    rprint(action_request)

==== 第一次 invoke 返回 ====
========= 原始响应 =========


{
    'messages': [
        HumanMessage(
            content="请帮我查询今天深圳的天气查询今天的新闻查看ID为 
'SK34132313210'的邮件内容向boy@qq.com发送邮件，标题是'关于AI编程',内容是: '你好，人工智能。'同时做这四件事",
            additional_kwargs={},
            response_metadata={},
            id='54b74cf2-37b0-4e8f-a565-254345948828'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 971,
                    'prompt_tokens': 626,
                    'total_tokens': 1597,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 818,
                        'rejected_prediction_tokens': None,
                        'text_tokens': 971
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 0,
                        'text_tokens': 626
                    }
                },
                'model_provider': 'openai',
                'model_name': 'qwen3.7-flash',
                'system_fingerprint': None,
                'id': 'chatcmpl-6f8f4d29-b77d-9ef8-ac2c-dfb76384a489',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a0e339-3b0e-7bf0-a987-bd907eacdae7-0',
            tool_calls=[
                {
                    'name': 'get_weather_tool',
                    'args': {'city': '深圳', 'is_forcast': False},
                    'id': 'call_95585b37dda143b499489cd8',
                    'type': 'tool_call'
                },
                {'name': 'get_news_tool', 'args': {}, 'id': 'call_22db3cc1c0a24e5f823a49e8', 'type': 'tool_call'},
                {
                    'name': 'read_email_tool',
                    'args': {'email_id': 'SK34132313210'},
                    'id': 'call_9859e7f575244713babac6a5',
                    'type': 'tool_call'
                },
                {
                    'name': 'send_email',
                    'args': {'recipient': 'boy@qq.com', 'subject': '关于AI编程', 'content': '你好，人工智能。'},
                    'id': 'call_9953288aff634af4973645c5',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 626,
                'output_tokens': 971,
                'total_tokens': 1597,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 818}
            }
        )
    ],
    '__interrupt__': [
        Interrupt(
            value={
                'action_requests': [
                    {
                        'name': 'get_weather_tool',
                        'args': {'city': '深圳', 'is_forcast': False},
                        'description': "中断咯\n\nTool: get_weather_tool\nArgs: {'city': '深圳', 'is_forcast': 
False}"
                    },
                    {
                        'name': 'get_news_tool',
                        'args': {},
                        'description': '中断咯\n\nTool: get_news_tool\nArgs: {}'
                    },
                    {
                        'name': 'send_email',
                        'args': {
                            'recipient': 'boy@qq.com',
                            'subject': '关于AI编程',
                            'content': '你好，人工智能。'
                        },
                        'description': '发送邮件中断啦'
                    }
                ],
                'review_configs': [
                    {
                        'action_name': 'get_weather_tool',
                        'allowed_decisions': ['approve', 'edit', 'reject', 'respond']
                    },
           

========= 美化输出 =========
================================ Human Message =================================

请帮我查询今天深圳的天气查询今天的新闻查看ID为 'SK34132313210'的邮件内容向boy@qq.com发送邮件，标题是'关于AI编程',内容是: '你好，人工智能。'同时做这四件事
================================== Ai Message ==================================
Tool Calls:
  get_weather_tool (call_95585b37dda143b499489cd8)
 Call ID: call_95585b37dda143b499489cd8
  Args:
    city: 深圳
    is_forcast: False
  get_news_tool (call_22db3cc1c0a24e5f823a49e8)
 Call ID: call_22db3cc1c0a24e5f823a49e8
  Args:
  read_email_tool (call_9859e7f575244713babac6a5)
 Call ID: call_9859e7f575244713babac6a5
  Args:
    email_id: SK34132313210
  send_email (call_9953288aff634af4973645c5)
 Call ID: call_9953288aff634af4973645c5
  Args:
    recipient: boy@qq.com
    subject: 关于AI编程
    content: 你好，人工智能。
============ interrupts ==============


[
    Interrupt(
        value={
            'action_requests': [
                {
                    'name': 'get_weather_tool',
                    'args': {'city': '深圳', 'is_forcast': False},
                    'description': "中断咯\n\nTool: get_weather_tool\nArgs: {'city': '深圳', 'is_forcast': False}"
                },
                {'name': 'get_news_tool', 'args': {}, 'description': '中断咯\n\nTool: get_news_tool\nArgs: {}'},
                {
                    'name': 'send_email',
                    'args': {'recipient': 'boy@qq.com', 'subject': '关于AI编程', 'content': '你好，人工智能。'},
                    'description': '发送邮件中断啦'
                }
            ],
            'review_configs': [
                {
                    'action_name': 'get_weather_tool',
                    'allowed_decisions': ['approve', 'edit', 'reject', 'respond']
                },
                {'action_name': 'get_news_tool', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']},
                {'action_name': 'send_email', 'allowed_decisions': ['approve', 'reject']}
            ]
        },
        id='dcf2a079ce33a1a8ac53a235d831924e'
    )
]

{
    'name': 'get_weather_tool',
    'args': {'city': '深圳', 'is_forcast': False},
    'description': "中断咯\n\nTool: get_weather_tool\nArgs: {'city': '深圳', 'is_forcast': False}"
}

{'name': 'get_news_tool', 'args': {}, 'description': '中断咯\n\nTool: get_news_tool\nArgs: {}'}

{
    'name': 'send_email',
    'args': {'recipient': 'boy@qq.com', 'subject': '关于AI编程', 'content': '你好，人工智能。'},
    'description': '发送邮件中断啦'
}

举例过程2：指明工具调用请求决策

In [12]:
from langgraph.types import Command

#如果有中断，说明进入了人在还了
weather_decision = {
    "type": "edit",
    "edited_action": {
        "name": "get_weather_tool",
        "args": {"city": "中国北京市", "is_forcast": True},
    }
}

news_decision = {
    "type": "approve",
}

send_email_decision = {
    "type": "approve",
}
decisions = {
    "decisions": []
}

#决策的顺序必须和返回的中断请求顺序一致
for action_request in action_requests:
    if action_request["name"] == "get_weather_tool":
        decisions["decisions"].append(weather_decision)
    if action_request["name"] == "get_news_tool":
        decisions["decisions"].append(news_decision)
    if action_request["name"] == "send_email":
        decisions["decisions"].append(send_email_decision)

if interrupts:
    #审批通过
    resumed_response = new_agent.invoke(
        Command(resume=decisions),
        config = custom_config,  #必须是在同一个 thread_id
    )
    print("========审批后继续执行=========")
    for msg in resumed_response["messages"]:
        msg.pretty_print()

>>> 真的执行发送邮件工具了
========审批后继续执行=========
================================ Human Message =================================

请帮我查询今天深圳的天气查询今天的新闻查看ID为 'SK34132313210'的邮件内容向boy@qq.com发送邮件，标题是'关于AI编程',内容是: '你好，人工智能。'同时做这四件事
================================== Ai Message ==================================
Tool Calls:
  get_weather_tool (call_95585b37dda143b499489cd8)
 Call ID: call_95585b37dda143b499489cd8
  Args:
    city: 中国北京市
    is_forcast: True
  get_news_tool (call_22db3cc1c0a24e5f823a49e8)
 Call ID: call_22db3cc1c0a24e5f823a49e8
  Args:
  read_email_tool (call_9859e7f575244713babac6a5)
 Call ID: call_9859e7f575244713babac6a5
  Args:
    email_id: SK34132313210
  send_email (call_9953288aff634af4973645c5)
 Call ID: call_9953288aff634af4973645c5
  Args:
    recipient: boy@qq.com
    subject: 关于AI编程
    content: 你好，人工智能。
================================= Tool Message =================================
Name: get_weather_tool

中国北京市今天天气不错
明天下雨
================================= Tool Message 